# 🎓 Assistant RAG IFOAD-UJKZ — Étape 1 : Ingestion & Vectorisation

**Projet Data Science 2026 — Master 1 IFOAD · Université Joseph Ki-Zerbo**

Ce notebook construit le **cœur du système RAG** :

| Étape | Description |
|-------|-------------|
| 0 | Installation des dépendances |
| 1 | Chargement des fichiers depuis Google Drive |
| 2 | Ingestion du calendrier Moodle (.ics) |
| 3 | Ingestion du CSV (maquette de cours) |
| 4 | Extraction du texte des PDF (avec OCR automatique pour les scans) |
| 5 | Chunking intelligent (découpage en segments contextualisés) |
| 6 | Génération des embeddings multilingues (Hugging Face, gratuit) |
| 7 | Stockage dans ChromaDB (base vectorielle persistante) |
| 8 | Tests de recherche sémantique |
| 9 | Sauvegarde de la base pour l'étape 2 |

---
**Structure attendue dans Google Drive (`My Drive > contents > pdf`) :**
```
📁 pdf/
  ├── CamScanner 08-07-2026 12.21.pdf
  ├── Recrutement master data science.pdf
  └── icalexport (1).ics
```

## 0. Installation des dépendances

In [1]:
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers chromadb pytesseract pdf2image

# Outils système nécessaires pour l'OCR (certains de vos PDF sont des scans/images)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra poppler-utils > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os
import shutil

PDF_DIR = "/content/pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# 🔒 Fichiers exclus pour confidentialité (contiennent des données personnelles d'étudiants)
FICHIERS_EXCLUS = [
    "Résultats_Recrutement_M1 Data Science (2).pdf",
    "Résultats_Recrutement_M1 Data Science.pdf",
]

DRIVE_DIR = "/content/drive/MyDrive/contents/pdf"  # ⚙️ Adaptez si besoin
ICS_PATH  = None
CSV_PATH  = None

# ─── Tentative Google Drive ────────────────────────────────────────────────
USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    USE_DRIVE = os.path.exists(DRIVE_DIR)
    if USE_DRIVE:
        print("✅ Google Drive monté.")
    else:
        print(f"⚠️  Drive monté mais '{DRIVE_DIR}' introuvable. On passe à l'upload direct.")
except Exception as e:
    print(f"⚠️  Drive non disponible ({e}). On passe à l'upload direct.")

if USE_DRIVE:
    for f in os.listdir(DRIVE_DIR):
        fp = os.path.join(DRIVE_DIR, f)
        if f.lower().endswith(".pdf") and f not in FICHIERS_EXCLUS:
            shutil.copy(fp, os.path.join(PDF_DIR, f))
        elif f in FICHIERS_EXCLUS:
            print(f"   🔒 '{f}' exclu (données personnelles).")
        elif f.endswith(".ics"):
            ICS_PATH = fp
        elif f.endswith(".csv"):
            CSV_PATH = fp
else:
    # ─── Upload direct via panneau 📁 ─────────────────────────────────────
    print("\n📋 Uploadez vos fichiers via le panneau 📁 (barre gauche de Colab) → ⬆️ Upload.")
    print("   Sélectionnez tous vos PDF + .ics + .csv, attendez la fin, puis relancez.\n")
    for f in os.listdir("/content"):
        fp = f"/content/{f}"
        if not os.path.isfile(fp):
            continue
        if f.lower().endswith(".pdf") and f not in FICHIERS_EXCLUS:
            shutil.copy(fp, os.path.join(PDF_DIR, f))
        elif f in FICHIERS_EXCLUS:
            print(f"   🔒 '{f}' exclu (données personnelles).")
        elif f.endswith(".ics") and ICS_PATH is None:
            ICS_PATH = fp
        elif f.endswith(".csv") and CSV_PATH is None:
            CSV_PATH = fp

# ─── Résumé ───────────────────────────────────────────────────────────────
pdfs = sorted(os.listdir(PDF_DIR))
print(f"\n📄 {len(pdfs)} PDF prêt(s) :")
for f in pdfs:
    print(f"   - {f}")
print("📅 ICS :", "✅ " + ICS_PATH if ICS_PATH else "❌ non trouvé")
print("📊 CSV :", "✅ " + CSV_PATH if CSV_PATH else "❌ non trouvé")
if not pdfs:
    print("\n⚠️  Aucun PDF trouvé. Uploadez vos fichiers et relancez cette cellule.")

Mounted at /content/drive
✅ Google Drive monté.

📄 2 PDF prêt(s) :
   - CamScanner 08-07-2026 12.21.pdf
   - Recrutement master data science.pdf
📅 ICS : ✅ /content/drive/MyDrive/contents/pdf/icalexport (1).ics
📊 CSV : ✅ /content/drive/MyDrive/contents/pdf/Curricula Data Science_IFOAD (2).csv


## 1. Chargement des fichiers depuis Google Drive

Vos fichiers sont dans **My Drive > contents > pdf**.  
Colab va demander une autorisation d'accès à Drive → acceptez.

In [3]:
import os
import shutil
from google.colab import drive

# ─── Montage de Google Drive ───────────────────────────────────────────────
drive.mount('/content/drive')

# ─── Chemins ──────────────────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/contents/pdf"
PDF_DIR   = "/content/pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# ─── Copie des PDF ────────────────────────────────────────────────────────
pdf_copied = 0
for f in os.listdir(DRIVE_DIR):
    if f.lower().endswith(".pdf"):
        shutil.copy(os.path.join(DRIVE_DIR, f), os.path.join(PDF_DIR, f))
        pdf_copied += 1

# ─── Chemins directs vers les autres fichiers ──────
ICS_PATH = os.path.join(DRIVE_DIR, "icalexport (1).ics")
CSV_PATH = os.path.join(DRIVE_DIR, "Curricula Data Science_IFOAD (2).csv")

# ─── Résumé ───────────────────────────────────────────────────────────────
print(f"✅ {pdf_copied} PDF copié(s) vers {PDF_DIR} :")
for f in sorted(os.listdir(PDF_DIR)):
    print(f"   📄 {f}")

print()
print("📅 Calendrier .ics :", "✅ trouvé" if os.path.exists(ICS_PATH) else "❌ non trouvé")
print("📊 Curricula CSV   :", "✅ trouvé" if os.path.exists(CSV_PATH) else "❌ non trouvé")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 2 PDF copié(s) vers /content/pdfs :
   📄 CamScanner 08-07-2026 12.21.pdf
   📄 Recrutement master data science.pdf

📅 Calendrier .ics : ✅ trouvé
📊 Curricula CSV   : ✅ trouvé


## 2. Ingestion du calendrier Moodle (.ics)

On parse les événements (examens, dépôts de projets, visioconférences) et on les convertit en chunks textuels.

In [4]:
import re
from datetime import datetime

def parse_ics_datetime(dt_str):
    dt_str = dt_str.strip()
    try:
        if "T" in dt_str:
            fmt = "%Y%m%dT%H%M%SZ" if dt_str.endswith("Z") else "%Y%m%dT%H%M%S"
        else:
            fmt = "%Y%m%d"
        dt = datetime.strptime(dt_str, fmt)
        return dt.strftime("%d/%m/%Y à %H:%M") if "T" in dt_str else dt.strftime("%d/%m/%Y")
    except Exception:
        return dt_str

def parse_ics(filepath):
    with open(filepath, encoding="utf-8", errors="ignore") as f:
        content = f.read()
    events = []
    for block in re.split(r"BEGIN:VEVENT", content)[1:]:
        def get(field):
            m = re.search(rf"{field}:(.+)", block)
            return m.group(1).strip().replace("\\n", " ").replace("\\,", ",") if m else ""
        summary     = get("SUMMARY")
        description = get("DESCRIPTION")
        dtstart     = parse_ics_datetime(get("DTSTART"))
        categories  = get("CATEGORIES")
        if summary:
            events.append({"summary": summary, "description": description,
                           "dtstart": dtstart, "categories": categories})
    return events

ics_chunks = []
if os.path.exists(ICS_PATH):
    events = parse_ics(ICS_PATH)
    for i, ev in enumerate(events):
        text = (
            f"[Calendrier académique IFOAD-UJKZ | Cours : {ev['categories']}]\n"
            f"Événement : {ev['summary']}\n"
            f"Date : {ev['dtstart']}\n"
        )
        if ev["description"]:
            text += f"Détails : {ev['description']}\n"
        ics_chunks.append({
            "id": f"ics_{i}",
            "text": text,
            "metadata": {
                "source_file": "icalexport.ics",
                "page": 1,
                "formation": "Master Sciences des Données (Data Science)",
                "type_doc": "Calendrier académique Moodle",
                "annee_academique": "2025-2026",
                "reference": ev["categories"],
            },
        })
    print(f"✅ {len(events)} événement(s) du calendrier chargé(s) :")
    for ev in events:
        print(f"   📅 {ev['dtstart']:22s} | {ev['categories']:20s} | {ev['summary']}")
else:
    print("⚠️ Fichier .ics non trouvé — vérifiez ICS_PATH.")

✅ 5 événement(s) du calendrier chargé(s) :
   📅 06/07/2026 à 00:00     | 1INF2102 -25-26      | Projet d'analyse de données Python 2026 doit être effectué
   📅 06/07/2026 à 00:00     | 1INF1102_1           | Projet final 2026 doit être effectué
   📅 08/07/2026 à 14:53     | Stat-Spa             | Visio est planifié pour
   📅 09/07/2026 à 23:00     | 1INF2202             | Examen final doit être effectué
   📅 15/07/2026 à 00:00     | EDD                  | Projet final sur les EDW doit être effectué


## 3. Ingestion du CSV (Curricula / Maquette de cours)

Le fichier `Curricula Data Science_IFOAD (2).csv` contient la maquette du programme. On le lit et on convertit chaque ligne en chunk textuel.

In [5]:
import os

csv_chunks = []

# Métadonnées globales du programme
programme = {
    "domaine": "Sciences et technologies",
    "mention": "Informatique",
    "specialite": "Science de données",
}

# TOUS LES COURS DU MASTER (M1S1, M1S2, M2S3, M2S4) REINSCRITS EN DUR
cours = [
    # ==========================================
    # MASTER 1 - SEMESTRE 1 (M1S1)
    # ==========================================
    {"semestre": "M1S1", "code_ue": "MTH2100", "ue": "Outils Mathématiques et statistiques I", "code_ec": "1MTH2100", "ec": "Probabilité et statistiques", "credits": 3},
    {"semestre": "M1S1", "code_ue": "MTH2100", "ue": "Outils Mathématiques et statistiques I", "code_ec": "2MTH2100", "ec": "Calcul matriciel numérique", "credits": 3},
    {"semestre": "M1S1", "code_ue": "MTH2101", "ue": "Outils Mathématiques et statistiques II", "code_ec": "1MTH2101", "ec": "Statistique inférentielle", "credits": 3},
    {"semestre": "M1S1", "code_ue": "MTH2101", "ue": "Outils Mathématiques et statistiques II", "code_ec": "2MTH2101", "ec": "Analyse de données", "credits": 3},
    {"semestre": "M1S1", "code_ue": "INF2100", "ue": "Base de données", "code_ec": "1INF2100", "ec": "UML", "credits": 2},
    {"semestre": "M1S1", "code_ue": "INF2100", "ue": "Base de données", "code_ec": "2INF2100", "ec": "Bases de données relationnelles", "credits": 3},
    {"semestre": "M1S1", "code_ue": "INF2101", "ue": "Programmation I", "code_ec": "1INF1102", "ec": "POO et Python", "credits": 3},
    {"semestre": "M1S1", "code_ue": "INF2101", "ue": "Programmation I", "code_ec": "2INF1102", "ec": "Logiciel R", "credits": 2},

    # ==========================================
    # MASTER 1 - SEMESTRE 2 (M1S2)
    # ==========================================
    {"semestre": "M1S2", "code_ue": "MTH2200", "ue": "Outils Mathématiques et statistiques III", "code_ec": "1MTH2200", "ec": "Séries temporelles", "credits": 3},
    {"semestre": "M1S2", "code_ue": "MTH2200", "ue": "Outils Mathématiques et statistiques III", "code_ec": "2MTH2200", "ec": "Statistique spatiale", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2200", "ue": "Machine Learning I", "code_ec": "1INF2200", "ec": "Introduction au machine learning", "credits": 2},
    {"semestre": "M1S2", "code_ue": "INF2200", "ue": "Machine Learning I", "code_ec": "2INF2200", "ec": "Machine learning supervisé", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2201", "ue": "Machine Learning II", "code_ec": "1INF2201", "ec": "Machine learning non supervisé", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2201", "ue": "Machine Learning II", "code_ec": "2INF2201", "ec": "Réseaux de neurones", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2202", "ue": "Programmation II", "code_ec": "1INF2202", "ec": "Programmation en Scala et PySpark", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2202", "ue": "Programmation II", "code_ec": "2INF2202", "ec": "Programmation en Julia", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2203", "ue": "Données massives II", "code_ec": "1INF2203", "ec": "Visualisation des données en R et Python", "credits": 3},
    {"semestre": "M1S2", "code_ue": "INF2203", "ue": "Données massives II", "code_ec": "2INF2203", "ec": "Projet Data Science", "credits": 2},
    {"semestre": "M1S2", "code_ue": "ANG2200", "ue": "Langue internationale", "code_ec": "1ANG2200", "ec": "Anglais", "credits": 2},

    # ==========================================
    # MASTER 2 - SEMESTRE 3 (M2S3)
    # ==========================================
    {"semestre": "M2S3", "code_ue": "INF2300", "ue": "Intelligence artificielle I", "code_ec": "1INF2300", "ec": "Machine learning", "credits": 2},
    {"semestre": "M2S3", "code_ue": "INF2300", "ue": "Intelligence artificielle I", "code_ec": "2INF2300", "ec": "Deep learning", "credits": 3},
    {"semestre": "M2S3", "code_ue": "INF2301", "ue": "Intelligence artificielle II", "code_ec": "1INF2301", "ec": "Applications du deep Learning", "credits": 2},
    {"semestre": "M2S3", "code_ue": "INF2301", "ue": "Intelligence artificielle II", "code_ec": "2INF2301", "ec": "Projet machine Learning", "credits": 2},
    {"semestre": "M2S3", "code_ue": "INF2302", "ue": "Données massives III", "code_ec": "1INF2302", "ec": "Technologie du big data", "credits": 3},
    {"semestre": "M2S3", "code_ue": "INF2302", "ue": "Données massives III", "code_ec": "2INF2302", "ec": "Base de données NoSQL", "credits": 3},
    {"semestre": "M2S3", "code_ue": "INF2303", "ue": "Données massives IV", "code_ec": "1INF2303", "ec": "Data visualisation", "credits": 3},
    {"semestre": "M2S3", "code_ue": "INF2303", "ue": "Données massives IV", "code_ec": "2INF2303", "ec": "Ethique de l'IA", "credits": 3},
    {"semestre": "M2S3", "code_ue": "INF2304", "ue": "Cloud computing", "code_ec": "1INF2304", "ec": "Introduction au cloud computing", "credits": 2},
    {"semestre": "M2S3", "code_ue": "INF2304", "ue": "Cloud computing", "code_ec": "2INF2304", "ec": "Virtualisation et conteneurisation", "credits": 3},
    {"semestre": "M2S3", "code_ue": "TCC2300", "ue": "Entrepreneuriat / Droit international", "code_ec": "1TCC2300", "ec": "Entrepreneuriat", "credits": 2},
    {"semestre": "M2S3", "code_ue": "TCC2300", "ue": "Entrepreneuriat / Droit international", "code_ec": "2TCC2300", "ec": "Aspect Juridique de la protection des données", "credits": 2},

    # ==========================================
    # MASTER 2 - SEMESTRE 4 (M2S4)
    # ==========================================
    {"semestre": "M2S4", "code_ue": "TCC2401", "ue": "Stage et soutenance", "code_ec": "1STG2400", "ec": "Stage, Rédaction du mémoire et soutenance", "credits": 30},
]

# Génération des chunks enrichis sémantiquement pour l'indexation vectorielle
for i, c in enumerate(cours):
    # Identification automatique du niveau académique pour la description du texte
    niveau = "Master 1" if "M1" in c["semestre"] else "Master 2"

    text = (
        f"[Maquette de cours IFOAD-UJKZ | Master Sciences des Données 2025-2026 | Semestre : {c['semestre']}]\n"
        f"Niveau : {niveau}\n"
        f"Unité d'enseignement (UE) : {c['ue']} (Code UE : {c['code_ue']})\n"
        f"Élément constitutif (EC) : {c['ec']} (Code EC : {c['code_ec']})\n"
        f"Crédits : {c['credits']} ECTS\n"
        f"Programme : Spécialité {programme['specialite']} (Mention {programme['mention']}), {programme['domaine']}"
    )

    csv_chunks.append({
        "id": f"csv_maquette_{c['semestre']}_{i}",
        "text": text,
        "metadata": {
            "source_file": "Curricula_Data_Science_IFOAD.csv",
            "page": 1,
            "formation": "Master Sciences des Données (Data Science)",
            "type_doc": "Maquette de cours",
            "annee_academique": "2025-2026",
            "reference": c["code_ec"],
            "semestre": c["semestre"],
            "code_ue": c["code_ue"]
        },
    })

print(f"✅ Opération réussie : {len(csv_chunks)} chunks sémantiques injectés en dur.")
print("\nAperçu d'un chunk du Master 2 (M2S3) :")
print(csv_chunks[23]["text"])

✅ Opération réussie : 32 chunks sémantiques injectés en dur.

Aperçu d'un chunk du Master 2 (M2S3) :
[Maquette de cours IFOAD-UJKZ | Master Sciences des Données 2025-2026 | Semestre : M2S3]
Niveau : Master 2
Unité d'enseignement (UE) : Données massives III (Code UE : INF2302)
Élément constitutif (EC) : Technologie du big data (Code EC : 1INF2302)
Crédits : 3 ECTS
Programme : Spécialité Science de données (Mention Informatique), Sciences et technologies


## 4. Extraction du texte des PDF (avec OCR automatique)

**Stratégie :** `pypdf` en premier (rapide). Si une page est vide (scan/image), bascule automatique sur Tesseract OCR en français.

In [6]:
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract

OCR_LANG         = "fra"   # Tesseract : langue française
OCR_DPI          = 300     # Résolution image (300 = bon équilibre qualité/vitesse)
MIN_CHARS_NATIVE = 20      # Seuil : moins de 20 caractères → page considérée comme scan

# ─── Métadonnées par fichier PDF ──────────────────────────────────────────
DOC_METADATA = {
    "Recrutement master data science.pdf": {
        "formation": "Master en Sciences des Données (Data Science)",
        "type_doc": "Communiqué de recrutement",
        "annee_academique": "2025-2026",
        "reference": "Communiqué N°2025-211/MESRI/SG/UJKZ/P",
    },
    "Résultats_Recrutement_M1 Data Science (2).pdf": {
        "formation": "Master en Sciences des Données (Data Science)",
        "type_doc": "Résultats de recrutement",
        "annee_academique": "2025-2026",
        "reference": "Résultats M1 Data Science",
    },
    "CamScanner 08-07-2026 12.21.pdf": {
        "formation": "Master en Sciences des Données (Data Science)",
        "type_doc": "Document scanné IFOAD",
        "annee_academique": "2025-2026",
        "reference": "CamScanner 08-07-2026",
    },
    # ── Anciens communiqués (si présents) ──
    "IFOAD-Communique-N_211-Master-Data-Science-2025-2026__1_.pdf": {
        "formation": "Master en Sciences des Données (Data Science)",
        "type_doc": "Communiqué de recrutement",
        "annee_academique": "2025-2026",
        "reference": "Communiqué N°2025-211/MESRI/SG/UJKZ/P",
    },
    "IFOA-Licdnce-en-informatique-applique.pdf": {
        "formation": "Licence en Informatique Appliquée (LIA)",
        "type_doc": "Communiqué de recrutement",
        "annee_academique": "2024-2025",
        "reference": "Communiqué N°2024-367/MESRI/SG/UJKZ/P",
    },
    "Recrutement-Licence-IFOAD-2022-2023.pdf": {
        "formation": "Licence de Sciences Informatiques Appliquées",
        "type_doc": "Communiqué de recrutement",
        "annee_academique": "2022-2023",
        "reference": "Communiqué N°2022-004/MESRI/SG/UJKZ/P",
    },
    "IFOAD-134-Appel-a-candidature-Certificat-Competences-aux-usages-du-numerique.pdf": {
        "formation": "Certificat en Compétences aux usages du numérique",
        "type_doc": "Appel à candidatures",
        "annee_academique": "2024-2025",
        "reference": "N°134/UJKZ/P/VPEIP",
    },
}

def get_metadata(filename):
    if filename in DOC_METADATA:
        return DOC_METADATA[filename]
    print(f"   ⚠️ Métadonnées non définies pour '{filename}' — utilisation des valeurs par défaut.")
    return {"formation": "IFOAD-UJKZ", "type_doc": "Document officiel",
            "annee_academique": "2025-2026", "reference": "N/A"}

def extract_pages(filepath):
    reader = PdfReader(filepath)
    n = len(reader.pages)
    native = [(reader.pages[i].extract_text() or "").strip() for i in range(n)]
    need_ocr = [i for i, t in enumerate(native) if len(t) < MIN_CHARS_NATIVE]
    pages = []
    if need_ocr:
        images = convert_from_path(filepath, dpi=OCR_DPI)
        for i in range(n):
            if i in need_ocr:
                text = pytesseract.image_to_string(images[i], lang=OCR_LANG).strip()
                if text:
                    pages.append({"page": i+1, "text": text, "source": "ocr"})
            elif native[i]:
                pages.append({"page": i+1, "text": native[i], "source": "pypdf"})
    else:
        pages = [{"page": i+1, "text": t, "source": "pypdf"}
                 for i, t in enumerate(native) if t]
    return pages

# ─── Extraction de tous les PDF ───────────────────────────────────────────
all_documents = []
for filename in sorted(os.listdir(PDF_DIR)):
    if not filename.lower().endswith(".pdf"):
        continue
    print(f"⏳ {filename}...")
    pages = extract_pages(os.path.join(PDF_DIR, filename))
    meta  = get_metadata(filename)
    all_documents.append({"filename": filename, "metadata": meta, "pages": pages})
    n_ocr = sum(1 for p in pages if p["source"] == "ocr")
    tag = f"(OCR sur {n_ocr}/{len(pages)} pages)" if n_ocr else "(texte natif)"
    print(f"   ✅ {len(pages)} page(s) extraite(s) {tag} | {meta['formation']}")

print(f"\n📄 {len(all_documents)} document(s) PDF traité(s).")

⏳ CamScanner 08-07-2026 12.21.pdf...
   ✅ 5 page(s) extraite(s) (OCR sur 5/5 pages) | Master en Sciences des Données (Data Science)
⏳ Recrutement master data science.pdf...
   ✅ 5 page(s) extraite(s) (OCR sur 5/5 pages) | Master en Sciences des Données (Data Science)

📄 2 document(s) PDF traité(s).


In [7]:
# Aperçu de la première page de chaque document
for doc in all_documents:
    print(f"\n{'='*65}")
    print(f"Fichier : {doc['filename']}")
    print(f"Type    : {doc['metadata']['type_doc']}")
    if doc["pages"]:
        p = doc["pages"][0]
        print(f"Source  : {p['source']} | Page {p['page']}")
        print(f"Extrait : {p['text'][:400]}")
    else:
        print("⚠️ Aucun texte extrait pour ce document.")


Fichier : CamScanner 08-07-2026 12.21.pdf
Type    : Document scanné IFOAD
Source  : ocr | Page 1
Extrait : BURKINA FASO

CELLELELELEE)
\ La Patrie ou la Mort, nous Vaincrons

MINISTERE DE L'ENSEIGNEMENT
SUPERIEUR, DE LA RECHERCHE
ET DE L'INNOVATION

.…——

SECRETARIAT GENERAL

 

Fort saPlENTA POPULO

._— 0 6 .
03 BP 7021 OUAGADOUGOU 03 JUIL 026
Tél. (226) 25 30 70 64/65 |
Fax. (226) 25 30 72 42
Télex : 5270 BF

*

Communiqué N° 2024 Q\ . /MESRI/SG/UJKZ/P
Portant recrutement en Licence d'Informatique Ap

Fichier : Recrutement master data science.pdf
Type    : Communiqué de recrutement
Source  : ocr | Page 1
Extrait : .

   

MINISTERE DE L'ENSEIGNEMENT FRA ' BURKINA FASO
SUPERIEUR, DE LA RECHERCHE 4 ù Li À Dé DÉS E
ET DE L'INNOVATION { y \\ La Patrie ou la Mort, nous Vaincrons

 

SECRETARIAT GENERAL
UNIVERSITE JOSEPH KI-ZERBO D mb Ecru
PRESIDENCE |
03 BP 7021 OUAGADOUGOU 03 Q 6 Juil 2026
Tél. (226) 25 30 70 64/65

Fax. (226) 25 30 72 42
Télex : 5270 BF

_©
\ Ÿ
Communiqué N° 2026 S°_/M

## 5. Chunking (découpage en segments contextualisés)

Chaque segment est **préfixé** par le nom de la formation et le type de document, pour que l'agent sache toujours de quelle source provient l'information.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

# chunk_size réduit à 400 pour mieux isoler chaque section (frais, conditions, etc.)
# chunk_overlap réduit à 80 pour conserver la continuité sans mélanger les sections
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def build_pdf_chunks(documents):
    chunks = []
    for doc in documents:
        meta = doc["metadata"]
        prefixe = (
            f"[Formation : {meta['formation']} | "
            f"Type : {meta['type_doc']} | "
            f"Année : {meta['annee_academique']}]\n"
        )
        for page in doc["pages"]:
            for split in text_splitter.split_text(page["text"]):
                chunks.append({
                    "id": f"chunk_{len(chunks)}",
                    "text": prefixe + split,
                    "metadata": {
                        "source_file": doc["filename"],
                        "page": page["page"],
                        "formation": meta["formation"],
                        "type_doc": meta["type_doc"],
                        "annee_academique": meta["annee_academique"],
                        "reference": meta["reference"],
                    },
                })
    return chunks

all_chunks = build_pdf_chunks(all_documents)

# ── Correction 3 : duplication des chunks CSV (×3) pour augmenter leur poids ──
# Les chunks CSV (maquette de cours) sont moins nombreux que les PDF.
# On les duplique pour qu'ils remontent mieux sur les questions de programme.
csv_chunks_weighted = []
for j, c in enumerate(csv_chunks):
    for k in range(3):  # chaque chunk CSV compte comme 3
        copy = dict(c)
        copy["id"] = f"csv_w_{j}_{k}"
        csv_chunks_weighted.append(copy)

# Fusion de toutes les sources
offset = len(all_chunks)
for j, c in enumerate(ics_chunks + csv_chunks_weighted):
    c["id"] = f"chunk_{offset + j}"
all_chunks.extend(ics_chunks)
all_chunks.extend(csv_chunks_weighted)

print(f"✅ {len(all_chunks)} chunks au total\n")
print("Répartition par type de source :")
for type_doc, count in Counter(c["metadata"]["type_doc"] for c in all_chunks).items():
    print(f"   {type_doc:45s} → {count} chunks")

✅ 147 chunks au total

Répartition par type de source :
   Document scanné IFOAD                         → 23 chunks
   Communiqué de recrutement                     → 23 chunks
   Calendrier académique Moodle                  → 5 chunks
   Maquette de cours                             → 96 chunks


In [9]:
# Aperçu d'un chunk PDF et d'un chunk calendrier
pdf_ex = next((c for c in all_chunks if c["metadata"]["type_doc"] != "Calendrier académique Moodle"
               and c["metadata"]["type_doc"] != "Maquette de cours"), None)
ics_ex = next((c for c in all_chunks if c["metadata"]["type_doc"] == "Calendrier académique Moodle"), None)
csv_ex = next((c for c in all_chunks if c["metadata"]["type_doc"] == "Maquette de cours"), None)

for label, ex in [("PDF", pdf_ex), ("Calendrier ICS", ics_ex), ("CSV Curricula", csv_ex)]:
    if ex:
        print(f"\n--- Exemple chunk {label} ---")
        print(ex["text"][:350])


--- Exemple chunk PDF ---
[Formation : Master en Sciences des Données (Data Science) | Type : Document scanné IFOAD | Année : 2025-2026]
BURKINA FASO

CELLELELELEE)
\ La Patrie ou la Mort, nous Vaincrons

MINISTERE DE L'ENSEIGNEMENT
SUPERIEUR, DE LA RECHERCHE
ET DE L'INNOVATION

.…——

SECRETARIAT GENERAL

 

Fort saPlENTA POPULO

._— 0 6 .
03 BP 7021 OUAGADOUGOU 03 JUIL 026

--- Exemple chunk Calendrier ICS ---
[Calendrier académique IFOAD-UJKZ | Cours : 1INF2102 -25-26]
Événement : Projet d'analyse de données Python 2026 doit être effectué
Date : 06/07/2026 à 00:00


--- Exemple chunk CSV Curricula ---
[Maquette de cours IFOAD-UJKZ | Master Sciences des Données 2025-2026 | Semestre : M1S1]
Niveau : Master 1
Unité d'enseignement (UE) : Outils Mathématiques et statistiques I (Code UE : MTH2100)
Élément constitutif (EC) : Probabilité et statistiques (Code EC : 1MTH2100)
Crédits : 3 ECTS
Programme : Spécialité Science de données (Mention Informatique


## 6. Génération des embeddings

Modèle : `paraphrase-multilingual-MiniLM-L12-v2` — gratuit, multilingue (français inclus), tourne sans GPU.

In [10]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print("⏳ Chargement du modèle (premier lancement = téléchargement ~470 Mo)...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Modèle chargé. Dimension des vecteurs : {embedding_model.get_sentence_embedding_dimension()}")

print(f"\n⏳ Génération des embeddings pour {len(all_chunks)} chunks...")
embeddings = embedding_model.encode(
    [c["text"] for c in all_chunks],
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"✅ Embeddings générés. Shape : {embeddings.shape}")

⏳ Chargement du modèle (premier lancement = téléchargement ~470 Mo)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Modèle chargé. Dimension des vecteurs : 384

⏳ Génération des embeddings pour 147 chunks...


/tmp/ipykernel_7651/4020447904.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Modèle chargé. Dimension des vecteurs : {embedding_model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Embeddings générés. Shape : (147, 384)


## 7. Stockage dans ChromaDB

In [11]:
import chromadb

CHROMA_DB_PATH = "/content/chroma_db_ifoad"
COLLECTION_NAME = "ifoad_communiques"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Repart propre à chaque exécution complète
try:
    client.delete_collection(COLLECTION_NAME)
    print("♻️  Ancienne collection supprimée.")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "Documents officiels et calendrier IFOAD-UJKZ"},
)

# Insertion par lots de 100 (évite les timeouts sur de grandes collections)
BATCH_SIZE = 100
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i+BATCH_SIZE]
    collection.add(
        ids       =[c["id"]       for c in batch],
        embeddings=[embeddings[i+j].tolist() for j in range(len(batch))],
        documents =[c["text"]     for c in batch],
        metadatas =[c["metadata"] for c in batch],
    )

print(f"✅ {collection.count()} chunks stockés dans '{COLLECTION_NAME}'")
print(f"📁 Base persistée dans : {CHROMA_DB_PATH}")

♻️  Ancienne collection supprimée.
✅ 147 chunks stockés dans 'ifoad_communiques'
📁 Base persistée dans : /content/chroma_db_ifoad


## 8. Tests de recherche sémantique

In [15]:
def search(query, n_results=3):
    vec = embedding_model.encode([query], normalize_embeddings=True)
    results = collection.query(query_embeddings=vec.tolist(), n_results=n_results)
    for i in range(len(results["ids"][0])):
        m = results["metadatas"][0][i]
        d = results["distances"][0][i]
        print(f"  [{i+1}] distance={d:.3f} | {m['type_doc']} | {m['formation'][:40]}")
        print(f"       {results['documents'][0][i][:250]}")
        print()

tests = [
    ("Frais d'inscription pour le Master Data Science",
     "→ doit remonter les communiqués de recrutement Master"),
    ("Quand est-ce que je dois rendre mon projet final ?",
     "→ doit remonter les événements du calendrier"),
    ("Quels cours sont au programme du Master au semestre 1 ?",
     "→ doit remonter la maquette CSV"),
    ("Météo à Ouagadougou demain",
     "→ question hors-sujet : distance élevée, l'agent dira 'je ne sais pas'"),
]

for query, expected in tests:
    print("="*65)
    print(f"❓ {query}")
    print(f"   {expected}")
    print()
    search(query)


❓ Frais d'inscription pour le Master Data Science
   → doit remonter les communiqués de recrutement Master

  [1] distance=0.485 | Communiqué de recrutement | Master en Sciences des Données (Data Sci
       [Formation : Master en Sciences des Données (Data Science) | Type : Communiqué de recrutement | Année : 2025-2026]
e maîtriser les outils et langages de la Data Science

:  Scanned with
: 9) CamScanner':

  [2] distance=0.534 | Communiqué de recrutement | Master en Sciences des Données (Data Sci
       [Formation : Master en Sciences des Données (Data Science) | Type : Communiqué de recrutement | Année : 2025-2026]
I Objectif de la formation

Le Master en Sciences des Données (Data Science) a pour ambition de former des
experts et des cadres supéri

  [3] distance=0.568 | Communiqué de recrutement | Master en Sciences des Données (Data Sci
       [Formation : Master en Sciences des Données (Data Science) | Type : Communiqué de recrutement | Année : 2025-2026]
Fax. (226) 25 30 72 42


## 9. Sauvegarde de la base ChromaDB

On zippe la base et on la sauvegarde directement dans Google Drive pour ne pas la perdre entre deux sessions Colab.

In [16]:
import shutil

# Sauvegarde dans Drive (recommandé — persistant)
DRIVE_SAVE_PATH = "/content/drive/MyDrive/contents/chroma_db_ifoad"
if os.path.exists(DRIVE_SAVE_PATH):
    shutil.rmtree(DRIVE_SAVE_PATH)
shutil.copytree(CHROMA_DB_PATH, DRIVE_SAVE_PATH)
print(f"✅ Base sauvegardée dans Google Drive : {DRIVE_SAVE_PATH}")

# Zip optionnel à télécharger localement
shutil.make_archive("/content/chroma_db_ifoad", "zip", CHROMA_DB_PATH)
print("✅ Archive zip créée : /content/chroma_db_ifoad.zip")
try:
    from google.colab import files
    files.download("/content/chroma_db_ifoad.zip")
except Exception:
    pass

✅ Base sauvegardée dans Google Drive : /content/drive/MyDrive/contents/chroma_db_ifoad
✅ Archive zip créée : /content/chroma_db_ifoad.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## ✅ Récapitulatif

| Source | Contenu | Chunks |
|--------|---------|--------|
| PDF (communiqués recrutement) | Conditions, frais, calendrier de candidature | Via OCR automatique |
| ICS (calendrier Moodle) | Examens, dépôts de projets, visioconférences | Événements formatés |
| CSV (curricula) | Maquette de cours, modules du programme | Ligne par ligne |
